# Pipeline 2 — CLIP ViT-B/32 (AI vs Real Detection)

This notebook evaluates the **CLIP ViT-B/32** pipeline for AI-generated image detection.  
Architecture: shared CLIP vision encoder with two independent linear classification heads (face + full image).  
Pretrained on **400M image-text pairs** by OpenAI, fine-tuned on GRAVEX-200K.

| Key Result | Value |
|---|---|
| Accuracy | **98.0%** |
| AUC-ROC | **0.998** |
| F1 Score | **0.979** |
| Parameters | ~87M |

---
## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Shared notebook utilities
import notebooks.nb_utils as nb_utils

# Pipeline
from inference.clip_pipeline import CLIPPipeline

import json
import numpy as np
import matplotlib.pyplot as plt

print(f"Project root: {PROJECT_ROOT}")
print("Setup complete.")

---
## 2. Dataset Overview (GRAVEX-200K)

The **GRAVEX-200K** dataset contains 200,000 images:
- **100,000 real** photographs
- **100,000 AI-generated** images
- Pre-split into **70% train / 20% validation / 10% test**
- Face crops pre-extracted with RetinaFace for the dual-branch architecture

In [ ]:
samples = nb_utils.show_dataset_overview()

---
## 3. Preprocessing Pipeline

CLIP ViT-B/32 uses **224x224** input resolution with CLIP/ImageNet normalization:
- `mean = [0.48145466, 0.4578275, 0.40821073]`
- `std  = [0.26862954, 0.26130258, 0.27577711]`

The three preprocessing steps are:
1. **Resize** — Bilinear/Bicubic/Lanczos interpolation to 224x224. Lanczos preserves the most detail.
2. **Color Space Analysis** — RGB channel decomposition reveals per-channel artifacts that AI generators often leave behind.
3. **Noise Reduction** — Gaussian, Median, and Bilateral filters can suppress sensor noise without destroying AI-specific patterns.

In [ ]:
# Pick a sample image for preprocessing visualization
sample_image_path = samples[0]["image_path"]
print(f"Sample image: {sample_image_path}")

nb_utils.show_preprocessing_steps(sample_image_path, target_size=224)

---
## 4. Augmentation Study

Augmentations serve two purposes for AI detection:
1. **Prevent shortcut learning** — without augmentation the model may learn to detect JPEG compression level, image resolution, or brightness as proxies for real vs AI.
2. **Simulate real-world degradation** — social media re-encoding, screenshots, and camera artifacts all modify images in ways the model must be robust to.

Each augmentation targets a specific failure mode:
- **Rotation/Flip/Crop/Translation** — geometric invariance; prevents positional bias
- **Blur** — forces model to use structure, not sharpness
- **Sharpening** — prevents over-reliance on edge softness (common in AI images)
- **Color Jitter** — prevents using white balance/exposure as a proxy
- **JPEG Compression** — prevents learning compression block boundaries as features

In [ ]:
nb_utils.show_augmentation_study(sample_image_path, size=224)

---
## 5. Model Architecture — CLIP ViT-B/32

### Architecture Overview

**CLIP ViT-B/32** is a Vision Transformer pretrained by OpenAI on **400 million image-text pairs** using contrastive learning. The model learns rich visual representations that transfer remarkably well to downstream tasks.

**ViT-B/32 Encoder Specs:**
- 12 transformer layers
- 768 hidden dimension
- 12 attention heads
- Patch size: 32x32 (input 224x224 yields 7x7 = 49 patches)
- ~87M total parameters

**Dual-Branch Design:**
```
                    +-----------+
   face_crop  ---> |           | ---> face_head (Linear 768->1) ---> face_logit
                    | Shared    |
                    | CLIP      |
                    | ViT-B/32  |
   full_image ---> |           | ---> full_head (Linear 768->1) ---> full_logit
                    +-----------+
```

**Key design decisions:**
- **Single shared encoder** processes both face crops and full images in one batched forward pass (face+full stacked along batch dim). This halves VRAM compared to two separate encoders.
- **Two independent linear heads** specialize: `face_head` detects face-specific AI artifacts, `full_head` detects global generation patterns.
- **Fusion at inference:** `0.6 x max(face_scores) + 0.4 x full_score`
- **Pretrained on 400M image-text pairs** — CLIP's contrastive pretraining learned texture, structure, and semantic features that are highly relevant to distinguishing AI-generated images from real photographs.

In [ ]:
from clip_detector.clip_dual_branch import CLIPDualBranchDetector

model = CLIPDualBranchDetector()
total = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for p in model.encoder.parameters())
print(f"Total parameters: {total:,}")
print(f"Shared encoder:   {encoder_params:,}")
print(f"Face head:        {sum(p.numel() for p in model.face_head.parameters()):,}")
print(f"Full head:        {sum(p.numel() for p in model.full_head.parameters()):,}")

# Clean up — we will use the pipeline (which loads fine-tuned weights) for inference
del model

---
## 6. Training Configuration

| Parameter | Value |
|---|---|
| Max Epochs | 30 |
| Batch Size (GPU) | 64 |
| Gradient Accumulation | 2 |
| Effective Batch Size | 128 |
| Learning Rate (heads) | 1e-4 |
| Learning Rate (encoder) | 1e-6 (lr x 0.01) |
| Optimizer | AdamW (betas=0.9/0.999, wd=1e-4) |
| Scheduler | Cosine Annealing + Warmup (2 epochs) |
| Loss | BCEWithLogitsLoss |
| Precision | 16-mixed (AMP) |
| Early Stopping | patience=7, monitor=val_fused_acc |
| Best Epoch | 25 (97.67% val_fused_acc) |

**Differential Learning Rate:** The encoder uses `lr x 0.01 = 1e-6` while classification heads use the full `lr = 1e-4`. This preserves CLIP's pretrained representations (learned from 400M pairs) while allowing the heads to rapidly specialize for AI detection.

In [ ]:
# Show training curves from val_metrics CSV if available
import pandas as pd

csv_path = PROJECT_ROOT / "results" / "val_metrics_clip.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curves
    if "val_loss" in df.columns:
        axes[0].plot(df["epoch"], df["val_loss"], "o-", color="#e74c3c", linewidth=2)
        axes[0].set_title("Validation Loss", fontweight="bold")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].grid(True, alpha=0.3)

    # Accuracy curves
    if "val_fused_acc" in df.columns:
        axes[1].plot(df["epoch"], df["val_fused_acc"], "o-", color="#2ecc71", linewidth=2)
        axes[1].set_title("Validation Fused Accuracy", fontweight="bold")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].grid(True, alpha=0.3)
        best_idx = df["val_fused_acc"].idxmax()
        axes[1].axvline(df.loc[best_idx, "epoch"], ls="--", color="gray", alpha=0.5)
        axes[1].annotate(
            f"Best: {df.loc[best_idx, 'val_fused_acc']:.4f} (epoch {int(df.loc[best_idx, 'epoch'])})",
            xy=(df.loc[best_idx, "epoch"], df.loc[best_idx, "val_fused_acc"]),
            fontsize=9, ha="center", va="bottom",
        )

    # Branch accuracies
    if "val_face_acc" in df.columns and "val_full_acc" in df.columns:
        axes[2].plot(df["epoch"], df["val_face_acc"], "o-", label="Face Branch", linewidth=2)
        axes[2].plot(df["epoch"], df["val_full_acc"], "s-", label="Full Branch", linewidth=2)
        axes[2].set_title("Branch Accuracies", fontweight="bold")
        axes[2].set_xlabel("Epoch")
        axes[2].set_ylabel("Accuracy")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

    plt.suptitle("CLIP ViT-B/32 — Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"  Loaded {len(df)} epochs from {csv_path.name}")
else:
    print(f"  Training CSV not found at {csv_path}")
    print("  (Training was done on RunPod; CSV may not have been copied back.)")
    print("  Best checkpoint: epoch 25, val_fused_acc = 0.9767")

---
## 7. Sample Predictions

Load the fine-tuned CLIP pipeline and visualize predictions on test images.

In [ ]:
pipeline = CLIPPipeline()

if pipeline.is_available():
    print(f"Pipeline: {pipeline.name}")
    print(f"Description: {pipeline.description}")
    print(f"Weights loaded successfully.")
else:
    print("WARNING: CLIP weights not found. Predictions will not be available.")
    print("Expected at: weights/clip_finetuned.pt")

In [ ]:
if pipeline.is_available():
    nb_utils.show_predictions(pipeline, samples, n=8)
else:
    print("Skipping predictions — weights not available.")

---
## 8. Evaluation Metrics

CLIP ViT-B/32 achieved **98% accuracy** on the test set with an **AUC-ROC of 0.998**, demonstrating near-perfect discrimination between real and AI-generated images.

In [ ]:
# Try to load pre-computed results; fall back to running evaluation
results_path = PROJECT_ROOT / "results" / "evaluation_clip" / "clip_eval_results.json"

if results_path.exists():
    with open(results_path) as f:
        saved_results = json.load(f)
    print("Loaded pre-computed evaluation results:")
    print(f"  Accuracy:   {saved_results['accuracy']:.4f}")
    print(f"  Precision:  {saved_results['precision']:.4f}")
    print(f"  Recall:     {saved_results['recall']:.4f}")
    print(f"  F1 Score:   {saved_results['f1']:.4f}")
    print(f"  AUC-ROC:    {saved_results['auc_roc']:.4f}")
else:
    print("Pre-computed results not found. Will run evaluation.")

In [ ]:
# Run full evaluation with confusion matrix and ROC curve
if pipeline.is_available():
    print("Running evaluation on test set...")
    y_true, y_scores = nb_utils.evaluate_pipeline(pipeline, samples, max_images=2000)
    clip_metrics = nb_utils.show_metrics(y_true, y_scores, model_name="CLIP ViT-B/32", threshold=0.5)
else:
    print("Pipeline not available — displaying saved metrics.")
    if results_path.exists():
        with open(results_path) as f:
            saved = json.load(f)
        print(f"\n  CLIP ViT-B/32 — Saved Evaluation Metrics")
        print(f"  {'─' * 50}")
        print(f"  Accuracy:   {saved['accuracy']:.4f}  ({saved['accuracy']:.1%})")
        print(f"  Precision:  {saved['precision']:.4f}")
        print(f"  Recall:     {saved['recall']:.4f}")
        print(f"  F1 Score:   {saved['f1']:.4f}")
        print(f"  AUC-ROC:    {saved['auc_roc']:.4f}")

---
## 9. Inference Speed

In [ ]:
if pipeline.is_available():
    nb_utils.show_inference_speed(pipeline, samples, n_runs=20)
else:
    print("Pipeline not available — skipping inference speed benchmark.")

---
## 10. Summary

**CLIP ViT-B/32 achieved 98% accuracy and 0.998 AUC-ROC** — far exceeding FerretNet's 50%.

CLIP's pretrained features from 400M diverse images transfer well to AI detection because modern AI generators produce images that differ from real photos in subtle texture/structure ways that CLIP's learned representations can capture. The differential learning rate (encoder lr x 0.01) preserves pretrained features while allowing heads to specialize. The dual-branch fusion (face + full image) provides robustness: face branch catches face-specific AI artifacts while full branch detects global generation patterns.

| Metric | CLIP ViT-B/32 |
|---|---|
| Accuracy | 98.0% |
| Precision | 98.7% |
| Recall | 97.1% |
| F1 Score | 0.979 |
| AUC-ROC | 0.998 |
| Parameters | ~87M |
| Input Size | 224x224 |
| Best Epoch | 25/30 |